# FM Change Detection — Colab quickstart

This notebook performs a clean clone, installs the project, runs the dataset-free smoke test, mounts LEVIR-CD from Google Drive, and runs a bounded four-representation benchmark. Select **Runtime → Change runtime type → GPU** before starting.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/fm-change-detection-benchmark.git"
BRANCH = "main"
assert "YOUR_USERNAME" not in REPO_URL, "Set REPO_URL to your pushed GitHub repository first."
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/fm-change-detection-benchmark
%cd /content/fm-change-detection-benchmark
!python -m pip install -q -e ".[dev]"

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), (
    "Use a GPU runtime for the real benchmark. The smoke test itself is CPU-safe."
)

In [ ]:
!fmcd smoke
!ruff check .
!ruff format --check .
!pytest -q

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DATA_ROOT = Path("/content/drive/MyDrive/datasets/LEVIR-CD")
required = [DATA_ROOT / name for name in ("A", "B", "label", "list")]
assert all(path.exists() for path in required), f"LEVIR-CD layout not found under {DATA_ROOT}"
!fmcd validate-data --root {DATA_ROOT}

In [ ]:
# Downloads pretrained weights on first use. Cached features make reruns resumable.
!fmcd benchmark --config configs/colab_quickstart.yaml --data-root {DATA_ROOT} --device cuda
!fmcd report --results outputs/colab_quickstart --output reports/colab_quickstart.md
from IPython.display import Markdown, display

display(Markdown(Path("reports/colab_quickstart.md").read_text()))